# 00 · 실행 환경 및 `data/raw` 데이터 검증 (로컬 / Colab 겸용)

### [0] 환경 부트스트랩 및 원본 파일 점검
- **데이터 범위**: `공통 환경 설정`
- **작업 단계**: `데이터 로딩 및 환경 검증`
- **목적 및 원리**: 로컬 VS Code(`.venv`)와 VS Code Colab 확장 커널(`ipykernel`)/Colab 웹 양쪽에서 `StopIteration` 없이 프로젝트 루트를 자동 탐색하고, 원본 CSV 9종과 핵심 패키지를 확인한다.
  - *주의 (VS Code Colab 커널)*: Colab 커널은 로컬 PC가 아닌 Google 클라우드 VM(`/content`)에서 실행되므로 로컬 폴더를 직접 읽지 못한다. 로컬 파일로 실행하려면 커널을 `.venv`로 선택하고, Colab 커널로 실행하려면 Google Drive에 `nfl-bdb2027` 저장소(및 데이터)를 두거나 `NFL_PROJECT_ROOT`를 지정해야 한다.
- **확인할 내용**: `9/9 CSVs ready`, `pyarrow`/`scikit-learn`/`tqdm` 정상 임포트.

In [ ]:
# =====================================================================
# [데이터: 공통] [단계: 환경 부트스트랩] 로컬/VS Code Colab 커널 겸용 루트 탐색
# =====================================================================
import importlib.util
import os
import sys
from pathlib import Path

# Google Drive 내 다른 경로에 저장소/데이터가 있다면 아래 주석을 풀고 경로를 지정하세요.
# os.environ["NFL_PROJECT_ROOT"] = "/content/drive/MyDrive/nfl-bdb2027"
# os.environ["NFL_DATA_DIR"] = "/content/drive/MyDrive/nfl-bdb2027-data"

def _is_colab_kernel() -> bool:
    if (
        "google.colab" in sys.modules
        or "COLAB_RELEASE_TAG" in os.environ
        or "COLAB_GPU" in os.environ
    ):
        return True
    try:
        return importlib.util.find_spec("google.colab") is not None
    except (ModuleNotFoundError, ValueError):
        return False

def find_project_root() -> Path:
    candidates: list[Path] = []
    if env_root := os.environ.get("NFL_PROJECT_ROOT"):
        candidates.append(Path(env_root).expanduser())

    cwd = Path.cwd()
    candidates.extend([cwd, *cwd.parents, cwd / "nfl-bdb2027"])

    in_colab = _is_colab_kernel()
    drive_root = Path("/content/drive")
    mydrive = drive_root / "MyDrive"

    if in_colab:
        if not mydrive.exists():
            try:
                from google.colab import drive
                drive.mount(str(drive_root), force_remount=False)
            except Exception as exc:
                print(f"[warn] Google Drive 자동 마운트 실패: {exc}")

        candidates.extend([
            Path("/content/nfl-bdb2027"),
            mydrive / "nfl-bdb2027",
            mydrive / "Colab Notebooks" / "nfl-bdb2027",
        ])

    seen: set[Path] = set()
    for p in candidates:
        if p in seen:
            continue
        seen.add(p)
        if (p / "src" / "config.py").exists():
            return p.resolve()

    if in_colab:
        skip = {"drive", "sample_data", ".config", ".git", ".venv", "__pycache__"}
        depth1_dirs: list[Path] = []
        for base in (Path("/content"), mydrive, drive_root / "Shareddrives"):
            if not base.exists():
                continue
            try:
                children = [
                    d for d in base.iterdir()
                    if d.is_dir() and d.name not in skip and not d.name.startswith(".")
                ]
            except OSError:
                continue
            depth1_dirs.extend(children)
            for d in children:
                for cand in (d, d / "nfl-bdb2027"):
                    if (cand / "src" / "config.py").exists():
                        return cand.resolve()

        for d in depth1_dirs:
            try:
                for sub in d.iterdir():
                    if not sub.is_dir() or sub.name.startswith("."):
                        continue
                    low = sub.name.lower()
                    if "nfl" in low or "bdb" in low:
                        if (sub / "src" / "config.py").exists():
                            return sub.resolve()
                    elif low in {"ml", "work", "projects", "repos", "github", "colab", "colab notebooks", "study"}:
                        cand = sub / "nfl-bdb2027"
                        if (cand / "src" / "config.py").exists():
                            return cand.resolve()
            except OSError:
                continue

    diag = [f"src/config.py를 찾지 못했습니다 (현재 커널 cwd={cwd}, colab={in_colab})."]
    if in_colab:
        for base in (Path("/content"), mydrive):
            if base.exists():
                try:
                    entries = sorted(p.name for p in base.iterdir() if not p.name.startswith("."))[:20]
                except OSError:
                    entries = ["<접근 불가>"]
                diag.append(f"  - {base} 폴더 목록: {entries}")
        diag.extend([
            "",
            "[원인] VS Code에서 Colab 커널(ipykernel)을 선택하면 코드는 로컬 PC가 아니라",
            "       원격 Google Colab VM(/content)에서 실행되므로 로컬 폴더를 직접 읽을 수 없습니다.",
            "[해결 방법]",
            "  1) 로컬 PC 파일로 실행: VS Code 우측 상단 커널을 Colab 대신 로컬 '.venv'로 변경",
            "  2) Colab 커널로 실행: Google Drive(/content/drive/MyDrive/nfl-bdb2027)에 저장소를 두거나,",
            '     다른 Drive 경로에 있다면 이 셀 상단에 os.environ["NFL_PROJECT_ROOT"] = "/content/drive/MyDrive/..." 지정',
        ])
    else:
        diag.append("환경변수 NFL_PROJECT_ROOT를 지정하거나 프로젝트 폴더 안에서 실행하세요.")
    raise FileNotFoundError("\n".join(diag))

ROOT = find_project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config
config.setup()

In [ ]:
# =====================================================================
# [데이터: 공통] [단계: 검증] data/raw CSV 9종 존재 여부 및 용량 확인
# =====================================================================
import pandas as pd

rows = []
for key, path in config.all_raw_files().items():
    exists = path.exists()
    size_mb = round(path.stat().st_size / (1024 ** 2), 2) if exists else 0.0
    rows.append({"key": key, "filename": path.name, "exists": exists, "size_mb": size_mb})

raw_status = pd.DataFrame(rows)
display(raw_status)
assert raw_status["exists"].all(), "누락된 원본 CSV 파일이 있습니다!"
print(f"[확인된 결과] 원본 CSV {len(raw_status)}종 모두 정상 연결 (총 {raw_status['size_mb'].sum():.1f} MB)")

In [ ]:
# =====================================================================
# [데이터: 공통] [단계: 검증] Parquet(pyarrow) · scikit-learn 임포트 확인
# =====================================================================
import pyarrow
import sklearn
import tqdm

print("[확인된 결과] pyarrow:", pyarrow.__version__, "| sklearn:", sklearn.__version__, "| tqdm:", tqdm.__version__)